[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Hosting an API &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up: the practice API, and `start`, `wait_until_up` and
`stop`, which run a start command as a host would. Run it first, then the tasks in order, since each
one uses the folder the tasks before it wrote. The last cell removes the folder.


In [1]:
import importlib
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from importlib.metadata import version
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier


def start(folder, command, environment):
    """Run a start command in a folder, as a host would, with these environment variables added."""
    variables = {name: value for name, value in {**os.environ, **environment}.items() if value is not None}
    return subprocess.Popen(f"exec {command}", shell=True, cwd=folder, env=variables,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)


def wait_until_up(address, path="/health", seconds=30):
    """The app's first answer at this path, or TimeoutError if nothing answers in that many seconds."""
    stop_at = time.monotonic() + seconds
    while time.monotonic() < stop_at:
        try:
            return requests.get(f"{address}{path}", timeout=2)
        except requests.ConnectionError:
            time.sleep(0.2)
    raise TimeoutError(f"nothing answered at {address}{path} within {seconds} seconds")


def stop(process):
    """Stop a started app, and wait until it has."""
    process.terminate()
    process.communicate(timeout=10)


BASE = practice_api.start()
print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** An app in a file.


In [2]:
HELLO = Path("scratch/hello-api")
HELLO.mkdir(parents=True, exist_ok=True)
(HELLO / "main.py").write_text("""from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def home():
    return {"hello": "stations"}
""")

print(sorted(path.name for path in HELLO.iterdir()))


['main.py']


`write_text` does what the notebook's `%%writefile` did: the file holds the app, and nothing runs it
yet.


**2.** Pinned requirements.


In [3]:
requirements = "".join(f"{package}=={version(package)}\n" for package in ["fastapi", "uvicorn"])
(HELLO / "requirements.txt").write_text(requirements)

print(requirements, end="")


fastapi==0.141.1
uvicorn==0.52.4


The versions come from the packages this notebook imported, so the host would install what these
tasks run.


**3.** The host's start command.


In [4]:
server = start(HELLO, "uvicorn main:app --host 127.0.0.1 --port $PORT", {"PORT": "8041"})
print(wait_until_up("http://127.0.0.1:8041", path="/").json())
stop(server)


{'hello': 'stations'}


This app has no `/health`, so `wait_until_up` asks for `/`, which is also the answer to print.


**4.** A value from the environment.


In [5]:
(HELLO / "main.py").write_text("""import os

from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def home():
    return {"hello": os.environ.get("GREETING", "stations")}
""")

server = start(HELLO, "uvicorn main:app --host 127.0.0.1 --port $PORT", {"PORT": "8041", "GREETING": "Tromso"})
print(wait_until_up("http://127.0.0.1:8041", path="/").json())
stop(server)


{'hello': 'Tromso'}


`os.environ.get` gives a default for a variable that is not set, where `os.environ[...]` raises
`KeyError`. A greeting can have a default. A key should not, which is why the notebook's `main.py`
uses `os.environ[...]`.


**5.** The documentation, from the running app.


In [6]:
server = start(HELLO, "uvicorn main:app --host 127.0.0.1 --port $PORT", {"PORT": "8041"})
wait_until_up("http://127.0.0.1:8041", path="/")
for path in ["/docs", "/openapi.json"]:
    print(path, requests.get(f"http://127.0.0.1:8041{path}", timeout=10).status_code)
print(requests.get("http://127.0.0.1:8041/openapi.json", timeout=10).json()["info"]["title"])
stop(server)


/docs 200
/openapi.json 200
FastAPI


The app has no title of its own, so its OpenAPI document carries FastAPI's default, `FastAPI`.


**6.** The files, packed.


In [7]:
archive = Path("scratch/hello-api.zip")
with zipfile.ZipFile(archive, "w") as packed:
    for name in ["main.py", "requirements.txt"]:
        packed.write(HELLO / name, arcname=name)

with zipfile.ZipFile(archive) as packed:
    print(packed.namelist())


['main.py', 'requirements.txt']


Only the two files named, and not the `__pycache__` folder that running the app left beside them.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Hosting an API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/18-hosting-an-api.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
